In [ ]:
# Kopmplette Training Pipeline für Unwetterwarnung
import os
from pathlib import Path

import hopsworks
import joblib
import numpy as np
import pandas as pd
from dotenv import load_dotenv


def load_project():
    project_root = Path.cwd().resolve()
    if not (project_root / ".env").exists():
        project_root = project_root.parent
    load_dotenv(project_root / ".env")

    api_key = os.getenv("HOPSWORKS_API_KEY")
    project_name = os.getenv("HOPSWORKS_PROJECT_NAME")
    if not api_key or not project_name:
        raise ValueError(
            "HOPSWORKS_API_KEY und HOPSWORKS_PROJECT_NAME müssen in der .env-Datei gesetzt sein."
        )

    return hopsworks.login(
        api_key_value=api_key,
        project=project_name,
        host="eu-west.cloud.hopsworks.ai",
        port=443,
    )


def run_training_pipeline(project, fg_name="weather_features_batch", fg_version=1):
    """End-to-End Training Pipeline für das Unwetterwarnungsmodell."""
    from sklearn.metrics import f1_score, roc_auc_score
    from sklearn.model_selection import train_test_split
    from xgboost import XGBClassifier

    feature_columns = [
        "temperature_2m", "relative_humidity_2m", "precipitation",
        "pressure_msl", "surface_pressure", "cloud_cover", "wind_speed_10m",
        "wind_gusts_10m", "cape", "precip_rolling_sum_3h",
        "precip_rolling_sum_6h", "precip_rolling_sum_12h", "wind_gust_max_3h",
        "wind_gust_max_6h", "wind_gust_max_12h", "pressure_mean_3h",
        "pressure_mean_6h", "pressure_mean_12h", "pressure_change_3h",
        "pressure_drop_rate", "wind_gust_anomaly", "temp_change_3h",
    ]
    label_column = "is_severe_weather"

    feature_store = project.get_feature_store()
    weather_fg = feature_store.get_feature_group(name=fg_name, version=fg_version)
    available_columns = {feature.name for feature in weather_fg.features}
    missing_columns = [
        column for column in feature_columns + [label_column]
        if column not in available_columns
    ]
    if missing_columns:
        raise ValueError(f"Fehlende Spalten in {fg_name}: {missing_columns}")

    feature_view = feature_store.get_or_create_feature_view(
        name="severe_weather_fv",
        version=1,
        description="Feature View für das Unwetterwarnungsmodell",
        query=weather_fg.select(feature_columns + [label_column]),
        labels=[label_column],
    )
    feature_view.create_training_data(
        data_format="csv", write_options={"wait_for_job": True}
    )
    X, y = feature_view.get_training_data(training_dataset_version=1)
    y[label_column] = pd.to_numeric(y[label_column], errors="coerce").fillna(0).astype(int)
    X = X[feature_columns].apply(pd.to_numeric, errors="coerce").fillna(0)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y[label_column], test_size=0.2, random_state=42, stratify=y[label_column]
    )
    model = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        eval_metric="aucpr", random_state=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test).astype(int)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    has_two_classes = np.unique(y_test).size == 2
    metrics = {
        "roc_auc": float(roc_auc_score(y_test, y_pred_proba)) if has_two_classes else 0.0,
        "f1_score": float(f1_score(y_test, y_pred, zero_division=0)),
    }
    if not has_two_classes:
        print("⚠️ ROC-AUC nicht berechnet: Im Testset ist nur eine Klasse vorhanden.")

    model_dir = "severe_weather_model"
    os.makedirs(model_dir, exist_ok=True)
    joblib.dump(model, os.path.join(model_dir, "model.joblib"))

    from hsml.model_schema import ModelSchema
    from hsml.schema import Schema

    model_schema = ModelSchema(
        input_schema=Schema(X_train),
        output_schema=Schema(pd.DataFrame({label_column: y_train})),
    )
    model_registry = project.get_model_registry()
    hw_model = model_registry.python.create_model(
        name="severe_weather_classifier",
        metrics=metrics,
        model_schema=model_schema,
        input_example=X_train.sample(1),
        description="XGBoost Sturmwarnung-Klassifikator",
        feature_view=feature_view,
    )
    hw_model.save(model_dir)

    print(f"✅ Pipeline abgeschlossen! Model v{hw_model.version} | Metriken: {metrics}")
    return hw_model, metrics


project = load_project()
model, metrics = run_training_pipeline(project)

2026-09-14 14:50:27,992 INFO: Closing external client and cleaning up certificates.
2026-09-14 14:50:27,995 INFO: Connection closed.
2026-09-14 14:50:27,997 INFO: Initializing external client
2026-09-14 14:50:27,998 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-09-14 14:50:28,598 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/44167


Training dataset job started successfully, you can follow the progress at 
http://eu-west.cloud.hopsworks.ai/p/44167/jobs/named/severe_weather_fv_1_create_fv_td_14092026145035/executions


2026-09-14 14:50:45,074 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-09-14 14:50:48,211 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-09-14 14:52:52,337 INFO: Waiting for execution to finish. Current state: FINISHED. Final status: SUCCEEDED
2026-09-14 14:52:52,803 INFO: Waiting for log aggregation to finish.
2026-09-14 14:52:52,804 INFO: Execution finished successfully.
2026-09-14 14:52:54,764 INFO: Provenance cached data - overwriting last accessed/created training dataset from 9 to 1.


⚠️ ROC-AUC nicht berechnet: Im Testset ist nur eine Klasse vorhanden.


Uploading model files (0 dirs, 0 files):  17%|█▋        | 1/6 [00:01<00:05,  1.19s/it]

Moving model files from 'severe_weather_model' to the model registry... This is the default behavior. Set keep_original_files=True to copy files instead.


Uploading /workspaces/fhnw_cas_aiops_project1_weather_forcasts/Trainings Pipeline/severe_weather_model/model.joblib: 100.000%|██████████| 150291/150291 elapsed<00:00 remaining<00:00
Uploading /workspaces/fhnw_cas_aiops_project1_weather_forcasts/Trainings Pipeline/severe_weather_model/metrics.json: 100.000%|██████████| 102/102 elapsed<00:00 remaining<00:00
Uploading /tmp/tmp3tdp4pyo/input_example.json: 100.000%|██████████| 256/256 elapsed<00:00 remaining<00:00
Uploading /tmp/tmp3tdp4pyo/model_schema.json: 100.000%|██████████| 1924/1924 elapsed<00:00 remaining<00:00
Model export complete: 100%|██████████| 6/6 [00:10<00:00,  1.81s/it]                   

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/44167/models/severe_weather_classifier/6
✅ Pipeline abgeschlossen! Model v6 | Metriken: {'roc_auc': 0.0, 'f1_score': 0.0}
